# 17. 境界修正と市場状態別の診断
出典: FX (2).ipynb、セルindex [36]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 36


In [ ]:
# ============================================================
# USD/JPY 15m
# CORRECTED NESTED WALK-FORWARD
# + MARKET REGIME ROBUSTNESS ANALYSIS
#
# ============================================================
#
# 今回の目的
# ------------------------------------------------------------
# 1. 年境界label leakageを除去
# 2. 欠測をまたぐ擬似30分取引を除去
# 3. Nested Walk-Forwardを再実行
# 4. 修正版OOS tradesだけを使って、
#
#    ・Trend
#    ・Volatility
#    ・Session
#    ・BUY / SELL
#
#    に利益が偏っていないか調べる
#
# ------------------------------------------------------------
# 重要:
#
# このコードではRegimeを使って取引を選ばない。
# Regime分析は「診断」のみ。
#
# Regimeを見てから同じTestデータ上で
# Filterを追加するとdata snoopingになる。
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)

TRADING_COST = 0.00004

THRESHOLDS = [
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

TREES = 250

RANDOM_STATE = 42

MIN_TRAIN_YEARS = 3

MIN_TRAIN_ROWS = 5000

MIN_EVAL_ROWS = 100

MIN_VALIDATION_TRADES = 100


# Regime結果を信用する最低trade数
MIN_REGIME_TRADES = 50

# 1年単位で安定性を評価する最低trade数
MIN_YEAR_REGIME_TRADES = 20


# Moving Block Bootstrap
N_BOOTSTRAP = 2000

BOOTSTRAP_BLOCK_LENGTH = 20


# 出力フォルダ
RUN_NAME = (
    "corrected_nested_regime_"
    +
    datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
)

OUTPUT_DIR = (
    Path.cwd()
    /
    RUN_NAME
)

OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. データ読み込み
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    index_col=0,
)


# タイムゾーンを明示的にUTCへ
df.index = pd.to_datetime(
    df.index,
    utc=True,
)


df.columns = [
    c.lower()
    for c in df.columns
]


required_columns = [
    "open",
    "high",
    "low",
    "close",
]


missing = [
    c
    for c in required_columns
    if c not in df.columns
]


if missing:

    raise ValueError(
        f"OHLC列がありません: {missing}"
    )


df = (
    df[
        required_columns
    ]
    .apply(
        pd.to_numeric,
        errors="raise"
    )
    .sort_index()
)


if not df.index.is_unique:

    raise ValueError(
        "timestampに重複があります"
    )


print(
    "===================================="
)

print(
    "DATA"
)

print(
    "===================================="
)

print(
    "Rows:",
    len(df)
)

print(
    "Start:",
    df.index.min()
)

print(
    "End:",
    df.index.max()
)


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = (
        close.diff()
    )

    gain = (
        delta.clip(
            lower=0
        )
    )

    loss = (
        -delta.clip(
            upper=0
        )
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1 + rs
        )
    )


# ============================================================
# 4. モデル用特徴量
#
# GitHub最新baselineと同じ構成
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",
]


def make_features(
    data
):

    x = (
        data.copy()
    )


    # ----------------------------------------
    # Return
    # ----------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (
            x["close"]
            .pct_change(n)
        )


    # ----------------------------------------
    # Volatility
    # ----------------------------------------

    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (
            x[
                "return_1"
            ]
            .rolling(n)
            .std()
        )


    # ----------------------------------------
    # Moving Average
    # ----------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (
            x["close"]
            .rolling(period)
            .mean()
        )

        x[
            f"ma{period}_distance"
        ] = (
            x["close"]
            /
            ma
            - 1
        )

        x[
            f"ma{period}_slope"
        ] = (
            ma.pct_change()
        )


    # ----------------------------------------
    # Candle
    # ----------------------------------------

    candle_range = (
        x["high"]
        -
        x["low"]
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        x["close"]
        -
        x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        -
        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )
    ) / candle_range

    x["lower_wick"] = (
        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )
        -
        x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"]
        -
        x["low"]
    ) / x["close"]


    # ----------------------------------------
    # RSI
    # ----------------------------------------

    x["rsi14"] = (
        calculate_rsi(
            x["close"],
            14
        )
        /
        100
    )


    # ----------------------------------------
    # ATR
    # ----------------------------------------

    previous_close = (
        x["close"]
        .shift(1)
    )

    true_range = (
        pd.concat(
            [

                x["high"]
                -
                x["low"],

                (
                    x["high"]
                    -
                    previous_close
                ).abs(),

                (
                    x["low"]
                    -
                    previous_close
                ).abs(),

            ],
            axis=1,
        )
        .max(
            axis=1
        )
    )

    x["atr14"] = (
        true_range
        .rolling(14)
        .mean()
        /
        x["close"]
    )


    # ----------------------------------------
    # High / Low
    # ----------------------------------------

    high16 = (
        x["high"]
        .rolling(16)
        .max()
    )

    low16 = (
        x["low"]
        .rolling(16)
        .min()
    )

    x[
        "distance_high_16"
    ] = (
        high16
        -
        x["close"]
    ) / x["close"]

    x[
        "distance_low_16"
    ] = (
        x["close"]
        -
        low16
    ) / x["close"]


    # ----------------------------------------
    # Time
    # ----------------------------------------

    hour = (
        x.index.hour
        +
        x.index.minute
        /
        60
    )

    x["hour_sin"] = np.sin(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["hour_cos"] = np.cos(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["weekday"] = (
        x.index.dayofweek
        /
        4
    )


    return x


# ============================================================
# 5. 正しいEntry / Exit / Label availability
#
# timestamp = 15分足のOPEN時刻
#
# t bar:
#   t ～ t+15分
#
# signal:
#   t bar close時点
#
# entry:
#   次足 t+1 のOpen
#
# exit:
#   t+2 bar のClose
#
# EntryからExitまでは30分
# ============================================================

def prepare_data(
    bars
):

    frame = (
        make_features(
            bars
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan
        )
    )


    times = pd.Series(
        bars.index,
        index=bars.index,
    )


    # 次足Open
    frame[
        "entry_time"
    ] = (
        times.shift(-1)
    )


    # t+2足のClose時刻
    frame[
        "label_end"
    ] = (
        times.shift(-2)
        +
        pd.Timedelta(
            minutes=15
        )
    )


    frame[
        "entry_price"
    ] = (
        bars[
            "open"
        ]
        .shift(-1)
    )


    frame[
        "exit_price"
    ] = (
        bars[
            "close"
        ]
        .shift(-2)
    )


    frame[
        "future_return"
    ] = (
        frame[
            "exit_price"
        ]
        /
        frame[
            "entry_price"
        ]
        - 1
    )


    frame[
        "target"
    ] = (
        frame[
            "future_return"
        ]
        > 0
    ).astype(int)


    # ========================================
    # 重要:
    #
    # t+1が本当に15分後か
    # t+2が本当に30分後か
    #
    # 欠測や週末跨ぎを除外する
    # ========================================

    continuous = (

        (
            times.shift(-1)
            -
            times
        )
        ==
        pd.Timedelta(
            minutes=15
        )

    ) & (

        (
            times.shift(-2)
            -
            times
        )
        ==
        pd.Timedelta(
            minutes=30
        )

    )


    frame = (
        frame.loc[
            continuous
        ]
        .dropna(
            subset=
                FEATURES
                +
                [
                    "future_return",
                    "label_end",
                    "entry_time",
                ]
        )
        .copy()
    )


    return frame


data = (
    prepare_data(
        df
    )
)


print()

print(
    "Usable rows after continuous-path check:",
    len(data)
)


# ============================================================
# 6. Regime用特徴量
#
# これらはモデルには入れない。
# あくまでOOS利益の発生源を診断する。
# ============================================================

def make_regime_features(
    bars
):

    r = pd.DataFrame(
        index=
            bars.index
    )


    ma20 = (
        bars[
            "close"
        ]
        .rolling(20)
        .mean()
    )

    ma100 = (
        bars[
            "close"
        ]
        .rolling(100)
        .mean()
    )


    previous_close = (
        bars[
            "close"
        ]
        .shift(1)
    )


    true_range = (
        pd.concat(
            [

                bars[
                    "high"
                ]
                -
                bars[
                    "low"
                ],

                (
                    bars[
                        "high"
                    ]
                    -
                    previous_close
                ).abs(),

                (
                    bars[
                        "low"
                    ]
                    -
                    previous_close
                ).abs(),

            ],
            axis=1
        )
        .max(
            axis=1
        )
    )


    atr_abs = (
        true_range
        .rolling(14)
        .mean()
    )


    # ----------------------------------------
    # Trend
    #
    # sign:
    # MA20 - MA100
    #
    # strength:
    # MA差をATRで標準化
    # ----------------------------------------

    r[
        "trend_distance"
    ] = (
        ma20
        -
        ma100
    )


    r[
        "trend_strength"
    ] = (
        (
            ma20
            -
            ma100
        )
        .abs()
        /
        atr_abs.replace(
            0,
            np.nan
        )
    )


    # ----------------------------------------
    # Volatility
    # ----------------------------------------

    r[
        "atr_pct"
    ] = (
        atr_abs
        /
        bars[
            "close"
        ]
    )


    # signalが利用可能になる時刻
    r[
        "signal_close_time"
    ] = (
        r.index
        +
        pd.Timedelta(
            minutes=15
        )
    )


    return r


regime_data = (
    make_regime_features(
        df
    )
)


# ============================================================
# 7. モデル
# ============================================================

def build_model():

    return (
        RandomForestClassifier(

            n_estimators=
                TREES,

            max_depth=
                8,

            min_samples_leaf=
                30,

            max_features=
                "sqrt",

            class_weight=
                "balanced",

            random_state=
                RANDOM_STATE,

            n_jobs=
                -1,
        )
    )


# ============================================================
# 8. Prediction
# ============================================================

def predict_frame(
    model,
    frame
):

    prob = (
        model.predict_proba(
            frame[
                FEATURES
            ]
        )
    )


    classes = list(
        model.classes_
    )


    if (
        0 not in classes
        or
        1 not in classes
    ):

        raise ValueError(
            "Training data must contain both classes"
        )


    p_up = (
        prob[
            :,
            classes.index(
                1
            )
        ]
    )


    result = frame[
        [
            "entry_time",
            "label_end",
            "entry_price",
            "exit_price",
            "future_return",
        ]
    ].copy()


    result[
        "p_up"
    ] = (
        p_up
    )


    result[
        "confidence"
    ] = np.maximum(
        p_up,
        1
        -
        p_up
    )


    result[
        "direction"
    ] = np.where(
        p_up
        >=
        0.5,
        "BUY",
        "SELL"
    )


    result[
        "direction_correct"
    ] = (
        (
            p_up
            >=
            0.5
        )
        ==
        (
            frame[
                "future_return"
            ]
            > 0
        )
    )


    result[
        "gross_return"
    ] = (
        frame[
            "future_return"
        ]
        *
        np.where(
            p_up
            >=
            0.5,
            1,
            -1
        )
    )


    return result


# ============================================================
# 9. Trade selection
#
# 1 position at a time
# ============================================================

def select_trades(
    predictions,
    threshold
):

    candidates = (
        predictions.loc[
            predictions[
                "confidence"
            ]
            >=
            threshold
        ]
        .sort_index()
    )


    selected = []

    next_allowed_entry = None


    for row in (
        candidates.itertuples()
    ):

        if (
            next_allowed_entry
            is not None
            and
            row.entry_time
            <
            next_allowed_entry
        ):

            continue


        selected.append(
            row.Index
        )


        # 前のtradeが完全に終了するまで
        # 新規Entry禁止
        next_allowed_entry = (
            row.label_end
        )


    trades = (
        candidates.loc[
            selected
        ]
        .copy()
    )


    trades[
        "net_return"
    ] = (
        trades[
            "gross_return"
        ]
        -
        TRADING_COST
    )


    trades.index.name = (
        "signal_time"
    )


    return trades


# ============================================================
# 10. Strategy statistics
# ============================================================

def profit_factor(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )


    gains = (
        r[
            r > 0
        ].sum()
    )

    losses = (
        -r[
            r < 0
        ].sum()
    )


    if losses > 0:

        return (
            gains
            /
            losses
        )


    if gains > 0:

        return np.inf


    return np.nan


def strategy_stats(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )


    if len(
        r
    ) == 0:

        return {

            "trades":
                0,

            "net_win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "median_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "growth":
                np.nan,

            "sum_return":
                np.nan,
        }


    equity = np.r_[
        1.0,
        np.cumprod(
            1
            +
            r
        )
    ]


    peak = (
        np.maximum.accumulate(
            equity
        )
    )


    dd = (
        equity
        /
        peak
        - 1
    )


    return {

        "trades":
            len(r),

        "net_win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            r.mean(),

        "median_return":
            np.median(
                r
            ),

        "profit_factor":
            profit_factor(
                r
            ),

        "max_dd":
            dd.min(),

        "growth":
            equity[-1]
            - 1,

        "sum_return":
            r.sum(),
    }


# ============================================================
# 11. ValidationでThresholdを選択
# ============================================================

def choose_threshold(
    validation_predictions
):

    rows = []

    best_threshold = None

    best_score = (
        -np.inf
    )


    for threshold in (
        THRESHOLDS
    ):

        trades = (
            select_trades(
                validation_predictions,
                threshold,
            )
        )


        stats = (
            strategy_stats(
                trades[
                    "net_return"
                ]
            )
        )


        eligible = (
            len(
                trades
            )
            >=
            MIN_VALIDATION_TRADES
        )


        if eligible:

            score = (
                stats[
                    "avg_return"
                ]
                *
                np.sqrt(
                    stats[
                        "trades"
                    ]
                )
            )

        else:

            score = (
                np.nan
            )


        rows.append(
            {

                "threshold":
                    threshold,

                "eligible":
                    eligible,

                "score":
                    score,

                **stats,
            }
        )


        if (
            eligible
            and
            score
            >
            best_score
        ):

            best_threshold = (
                threshold
            )

            best_score = (
                score
            )


    return (
        best_threshold,
        pd.DataFrame(
            rows
        )
    )


# ============================================================
# 12. Corrected Annual Nested Walk-Forward
#
# 年境界でlabel_endをpurge
# ============================================================

years = sorted(
    data.index.year.unique()
)


annual_rows = []

all_test_trades = []

validation_tables = []


for test_year in (
    years
):

    validation_year = (
        test_year
        -
        1
    )


    previous_years = [
        y
        for y in years
        if y
        <
        validation_year
    ]


    if (
        len(
            previous_years
        )
        <
        MIN_TRAIN_YEARS
    ):

        continue


    if (
        validation_year
        not in years
    ):

        continue


    validation_start = pd.Timestamp(
        year=
            validation_year,
        month=
            1,
        day=
            1,
        tz=
            "UTC",
    )


    test_start = pd.Timestamp(
        year=
            test_year,
        month=
            1,
        day=
            1,
        tz=
            "UTC",
    )


    test_end = pd.Timestamp(
        year=
            test_year
            +
            1,
        month=
            1,
        day=
            1,
        tz=
            "UTC",
    )


    # ========================================
    # Train
    # label結果がValidation開始までに判明済み
    # ========================================

    train = (
        data.loc[
            (
                data.index
                <
                validation_start
            )
            &
            (
                data[
                    "label_end"
                ]
                <=
                validation_start
            )
        ]
        .copy()
    )


    # ========================================
    # Validation
    # 結果がTest開始前に判明済み
    # ========================================

    validation = (
        data.loc[
            (
                data.index
                >=
                validation_start
            )
            &
            (
                data.index
                <
                test_start
            )
            &
            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )
        ]
        .copy()
    )


    # ========================================
    # Final Train
    # Test直前まで
    # ========================================

    final_train = (
        data.loc[
            (
                data.index
                <
                test_start
            )
            &
            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )
        ]
        .copy()
    )


    # ========================================
    # Test
    # ========================================

    test = (
        data.loc[
            (
                data.index
                >=
                test_start
            )
            &
            (
                data.index
                <
                test_end
            )
            &
            (
                data[
                    "label_end"
                ]
                <=
                test_end
            )
        ]
        .copy()
    )


    if (

        len(
            train
        )
        <
        MIN_TRAIN_ROWS

        or

        len(
            final_train
        )
        <
        MIN_TRAIN_ROWS

        or

        len(
            validation
        )
        <
        MIN_EVAL_ROWS

        or

        len(
            test
        )
        <
        MIN_EVAL_ROWS

    ):

        continue


    print()

    print(
        "===================================="
    )

    print(
        f"CORRECTED TEST YEAR {test_year}"
    )

    print(
        "===================================="
    )


    # ========================================
    # Train → Validation
    # ========================================

    model = (
        build_model()
    )


    model.fit(
        train[
            FEATURES
        ],
        train[
            "target"
        ]
    )


    validation_predictions = (
        predict_frame(
            model,
            validation
        )
    )


    (
        selected_threshold,
        threshold_table,
    ) = (
        choose_threshold(
            validation_predictions
        )
    )


    threshold_table[
        "test_year"
    ] = (
        test_year
    )


    validation_tables.append(
        threshold_table
    )


    if (
        selected_threshold
        is None
    ):

        print(
            "No eligible threshold"
        )

        continue


    # ========================================
    # Train + Validationで再学習
    # ========================================

    final_model = (
        build_model()
    )


    final_model.fit(
        final_train[
            FEATURES
        ],
        final_train[
            "target"
        ]
    )


    test_predictions = (
        predict_frame(
            final_model,
            test
        )
    )


    # ========================================
    # Test AUC
    # ========================================

    test_auc = (
        roc_auc_score(
            test[
                "target"
            ],
            test_predictions[
                "p_up"
            ]
        )
    )


    # ========================================
    # Validationで選んだThresholdを固定
    # ========================================

    selected = (
        select_trades(
            test_predictions,
            selected_threshold,
        )
    )


    selected[
        "test_year"
    ] = (
        test_year
    )


    selected[
        "threshold"
    ] = (
        selected_threshold
    )


    all_test_trades.append(
        selected
    )


    stats = (
        strategy_stats(
            selected[
                "net_return"
            ]
        )
    )


    annual_rows.append(
        {

            "test_year":
                test_year,

            "validation_year":
                validation_year,

            "threshold":
                selected_threshold,

            "test_auc":
                test_auc,

            "direction_accuracy":
                selected[
                    "direction_correct"
                ].mean(),

            **stats,
        }
    )


    print(
        "Threshold:",
        selected_threshold
    )

    print(
        "Test AUC:",
        round(
            test_auc,
            4
        )
    )

    print(
        "Trades:",
        stats[
            "trades"
        ]
    )

    print(
        "Avg Return:",
        round(
            stats[
                "avg_return"
            ]
            *
            100,
            6
        ),
        "%"
    )

    print(
        "PF:",
        round(
            stats[
                "profit_factor"
            ],
            4
        )
    )


# ============================================================
# 13. Corrected Nested結果
# ============================================================

annual_results = (
    pd.DataFrame(
        annual_rows
    )
)


trades = (
    pd.concat(
        all_test_trades
    )
    .sort_index()
)


overall = (
    strategy_stats(
        trades[
            "net_return"
        ]
    )
)


print()

print(
    "===================================="
)

print(
    "CORRECTED NESTED BASELINE"
)

print(
    "===================================="
)


annual_show = (
    annual_results
    .copy()
)


for col in [
    "threshold",
    "direction_accuracy",
    "net_win_rate",
    "avg_return",
    "median_return",
    "max_dd",
    "growth",
]:

    if col in (
        annual_show.columns
    ):

        annual_show[
            col
        ] *= 100


print(
    annual_show.to_string(
        index=False
    )
)


print()

print(
    "Overall OOS"
)

print(
    "Trades:",
    overall[
        "trades"
    ]
)

print(
    "Net win rate:",
    overall[
        "net_win_rate"
    ]
    *
    100,
    "%"
)

print(
    "Avg Return:",
    overall[
        "avg_return"
    ]
    *
    100,
    "%"
)

print(
    "PF:",
    overall[
        "profit_factor"
    ]
)

print(
    "Max DD:",
    overall[
        "max_dd"
    ]
    *
    100,
    "%"
)

print(
    "Growth:",
    overall[
        "growth"
    ]
    *
    100,
    "%"
)


# ============================================================
# 14. Regime thresholdを
# Test年より前だけから決める
#
# 未来のTest年を使ってRegime境界を作らない
# ============================================================

def get_regime_cutoffs(
    test_year
):

    test_start = pd.Timestamp(
        year=
            test_year,
        month=
            1,
        day=
            1,
        tz=
            "UTC",
    )


    historical = (
        regime_data.loc[
            regime_data[
                "signal_close_time"
            ]
            <=
            test_start
        ]
        .dropna(
            subset=[
                "trend_strength",
                "atr_pct",
            ]
        )
    )


    if len(
        historical
    ) < 1000:

        raise ValueError(
            "Regime historical sample too small"
        )


    return {

        # bottom 1/3 = Range
        "trend_q33":
            historical[
                "trend_strength"
            ].quantile(
                1/3
            ),

        # Volatility tertiles
        "vol_q33":
            historical[
                "atr_pct"
            ].quantile(
                1/3
            ),

        "vol_q67":
            historical[
                "atr_pct"
            ].quantile(
                2/3
            ),
    }


# ============================================================
# 15. OOS tradesへRegimeを付与
# ============================================================

regime_trade_frames = []

cutoff_rows = []


for test_year in sorted(
    trades[
        "test_year"
    ].unique()
):

    year_trades = (
        trades.loc[
            trades[
                "test_year"
            ]
            ==
            test_year
        ]
        .copy()
    )


    cutoffs = (
        get_regime_cutoffs(
            int(
                test_year
            )
        )
    )


    cutoff_rows.append(
        {

            "test_year":
                test_year,

            **cutoffs,
        }
    )


    regime_values = (
        regime_data.reindex(
            year_trades.index
        )
    )


    year_trades[
        "trend_strength"
    ] = (
        regime_values[
            "trend_strength"
        ]
    )


    year_trades[
        "trend_distance"
    ] = (
        regime_values[
            "trend_distance"
        ]
    )


    year_trades[
        "atr_pct"
    ] = (
        regime_values[
            "atr_pct"
        ]
    )


    # ========================================
    # Trend Regime
    #
    # 過去データのbottom 1/3をRange
    # それ以上はMA20-MA100の符号で
    # Up / Down
    # ========================================

    weak_trend = (
        year_trades[
            "trend_strength"
        ]
        <=
        cutoffs[
            "trend_q33"
        ]
    )


    year_trades[
        "trend_regime"
    ] = np.where(

        weak_trend,

        "RANGE",

        np.where(

            year_trades[
                "trend_distance"
            ]
            >
            0,

            "UP_TREND",

            "DOWN_TREND",
        )
    )


    # ========================================
    # Volatility Regime
    # ========================================

    year_trades[
        "vol_regime"
    ] = np.select(

        [

            year_trades[
                "atr_pct"
            ]
            <=
            cutoffs[
                "vol_q33"
            ],

            year_trades[
                "atr_pct"
            ]
            <=
            cutoffs[
                "vol_q67"
            ],

        ],

        [

            "LOW_VOL",
            "MID_VOL",

        ],

        default=
            "HIGH_VOL",
    )


    # ========================================
    # Session
    #
    # entry_timeをUTCで固定bucket化
    #
    # DSTの影響を避けるため、
    # 厳密な取引所session名ではなく
    # UTC時間帯として扱う。
    # ========================================

    entry_hour = (
        year_trades[
            "entry_time"
        ].dt.hour
    )


    year_trades[
        "session_block"
    ] = np.select(

        [

            (
                entry_hour
                >=
                0
            )
            &
            (
                entry_hour
                <
                8
            ),

            (
                entry_hour
                >=
                8
            )
            &
            (
                entry_hour
                <
                13
            ),

            (
                entry_hour
                >=
                13
            )
            &
            (
                entry_hour
                <
                21
            ),

        ],

        [

            "UTC_00_08",
            "UTC_08_13",
            "UTC_13_21",

        ],

        default=
            "UTC_21_24",
    )


    regime_trade_frames.append(
        year_trades
    )


regime_trades = (
    pd.concat(
        regime_trade_frames
    )
    .sort_index()
)


regime_cutoffs = (
    pd.DataFrame(
        cutoff_rows
    )
)


# ============================================================
# 16. Group statistics
# ============================================================

def group_statistics(
    frame,
    group_column
):

    rows = []


    for group_name, group in (
        frame.groupby(
            group_column
        )
    ):

        stats = (
            strategy_stats(
                group[
                    "net_return"
                ]
            )
        )


        rows.append(
            {

                group_column:
                    group_name,

                "direction_accuracy":
                    group[
                        "direction_correct"
                    ].mean(),

                **stats,
            }
        )


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "profit_factor",
            ascending=False
        )
    )


trend_results = (
    group_statistics(
        regime_trades,
        "trend_regime"
    )
)


vol_results = (
    group_statistics(
        regime_trades,
        "vol_regime"
    )
)


session_results = (
    group_statistics(
        regime_trades,
        "session_block"
    )
)


side_results = (
    group_statistics(
        regime_trades,
        "direction"
    )
)


# ============================================================
# 17. Trend × Volatility
# ============================================================

cross_rows = []


for (
    trend_name,
    vol_name
), group in (

    regime_trades.groupby(
        [
            "trend_regime",
            "vol_regime",
        ]
    )

):

    stats = (
        strategy_stats(
            group[
                "net_return"
            ]
        )
    )


    cross_rows.append(
        {

            "trend_regime":
                trend_name,

            "vol_regime":
                vol_name,

            "direction_accuracy":
                group[
                    "direction_correct"
                ].mean(),

            **stats,
        }
    )


trend_vol_results = (
    pd.DataFrame(
        cross_rows
    )
)


# ============================================================
# 18. 年別Regime安定性
# ============================================================

def regime_year_stability(
    frame,
    group_column
):

    yearly_rows = []


    for (
        group_name,
        year
    ), group in (

        frame.groupby(
            [
                group_column,
                "test_year",
            ]
        )

    ):

        if (
            len(
                group
            )
            <
            MIN_YEAR_REGIME_TRADES
        ):

            continue


        stats = (
            strategy_stats(
                group[
                    "net_return"
                ]
            )
        )


        yearly_rows.append(
            {

                group_column:
                    group_name,

                "test_year":
                    year,

                **stats,
            }
        )


    yearly = (
        pd.DataFrame(
            yearly_rows
        )
    )


    if yearly.empty:

        return (
            yearly,
            pd.DataFrame()
        )


    stability = (

        yearly
        .groupby(
            group_column
        )
        .agg(

            years_evaluated=(
                "test_year",
                "nunique"
            ),

            positive_years=(
                "avg_return",
                lambda x:
                    (
                        x > 0
                    ).sum()
            ),

            pf_above_1_years=(
                "profit_factor",
                lambda x:
                    (
                        x > 1
                    ).sum()
            ),

            median_year_pf=(
                "profit_factor",
                "median"
            ),

            minimum_year_pf=(
                "profit_factor",
                "min"
            ),

            median_year_return=(
                "avg_return",
                "median"
            ),

        )
        .reset_index()
    )


    return (
        yearly,
        stability
    )


trend_yearly, trend_stability = (
    regime_year_stability(
        regime_trades,
        "trend_regime"
    )
)


vol_yearly, vol_stability = (
    regime_year_stability(
        regime_trades,
        "vol_regime"
    )
)


session_yearly, session_stability = (
    regime_year_stability(
        regime_trades,
        "session_block"
    )
)


side_yearly, side_stability = (
    regime_year_stability(
        regime_trades,
        "direction"
    )
)


# ============================================================
# 19. Moving Block Bootstrap
#
# tradeを完全shuffleしない。
# 時系列依存を少し残したまま
# 平均ReturnのCIを作る。
# ============================================================

def moving_block_bootstrap(
    returns,
    n_bootstrap=N_BOOTSTRAP,
    block_length=BOOTSTRAP_BLOCK_LENGTH,
    seed=42,
):

    x = np.asarray(
        returns,
        dtype=float
    )


    n = len(
        x
    )


    if (
        n
        <
        max(
            30,
            block_length
        )
    ):

        return {

            "bootstrap_mean":
                np.nan,

            "ci_2_5":
                np.nan,

            "ci_97_5":
                np.nan,

            "prob_mean_positive":
                np.nan,
        }


    rng = np.random.default_rng(
        seed
    )


    max_start = (
        n
        -
        block_length
    )


    n_blocks = int(
        np.ceil(
            n
            /
            block_length
        )
    )


    bootstrap_means = np.empty(
        n_bootstrap
    )


    for i in range(
        n_bootstrap
    ):

        starts = (
            rng.integers(
                0,
                max_start
                +
                1,
                size=
                    n_blocks,
            )
        )


        sample = np.concatenate(
            [
                x[
                    s:
                    s
                    +
                    block_length
                ]
                for s in starts
            ]
        )[:n]


        bootstrap_means[
            i
        ] = (
            sample.mean()
        )


    return {

        "bootstrap_mean":
            x.mean(),

        "ci_2_5":
            np.quantile(
                bootstrap_means,
                0.025
            ),

        "ci_97_5":
            np.quantile(
                bootstrap_means,
                0.975
            ),

        "prob_mean_positive":
            (
                bootstrap_means
                >
                0
            ).mean(),
    }


# ============================================================
# 20. Bootstrap: 全体 + Regime
# ============================================================

bootstrap_rows = []


# 全体
bootstrap_rows.append(
    {

        "dimension":
            "OVERALL",

        "group":
            "ALL",

        "trades":
            len(
                regime_trades
            ),

        **moving_block_bootstrap(
            regime_trades[
                "net_return"
            ]
        ),
    }
)


for dimension in [

    "trend_regime",
    "vol_regime",
    "session_block",
    "direction",

]:

    for group_name, group in (

        regime_trades.groupby(
            dimension
        )

    ):

        bootstrap_rows.append(
            {

                "dimension":
                    dimension,

                "group":
                    group_name,

                "trades":
                    len(
                        group
                    ),

                **moving_block_bootstrap(
                    group[
                        "net_return"
                    ]
                ),
            }
        )


bootstrap_results = (
    pd.DataFrame(
        bootstrap_rows
    )
)


# ============================================================
# 21. Leave-One-Regime-Out
#
# 1つのRegimeを全部消したら
# 戦略全体が壊れるかを見る。
#
# 例:
# HIGH_VOLを除外した瞬間PF<1
# → 利益がHIGH_VOLへ集中している可能性
# ============================================================

leave_one_out_rows = []


for dimension in [

    "trend_regime",
    "vol_regime",
    "session_block",
    "direction",

]:

    for group_name in (
        regime_trades[
            dimension
        ].dropna().unique()
    ):

        remaining = (
            regime_trades.loc[
                regime_trades[
                    dimension
                ]
                !=
                group_name
            ]
        )


        stats = (
            strategy_stats(
                remaining[
                    "net_return"
                ]
            )
        )


        leave_one_out_rows.append(
            {

                "dimension":
                    dimension,

                "removed_group":
                    group_name,

                **stats,
            }
        )


leave_one_out_results = (
    pd.DataFrame(
        leave_one_out_rows
    )
)


# ============================================================
# 22. 表示用関数
# ============================================================

def display_regime_table(
    title,
    table
):

    show = (
        table.copy()
    )


    percent_columns = [

        "direction_accuracy",
        "net_win_rate",
        "avg_return",
        "median_return",
        "max_dd",
        "growth",
        "sum_return",

    ]


    for col in (
        percent_columns
    ):

        if col in (
            show.columns
        ):

            show[
                col
            ] *= 100


    print()

    print(
        "===================================="
    )

    print(
        title
    )

    print(
        "===================================="
    )

    print(
        show.to_string(
            index=False
        )
    )


# ============================================================
# 23. 結果表示
# ============================================================

display_regime_table(
    "TREND REGIME",
    trend_results,
)


display_regime_table(
    "VOLATILITY REGIME",
    vol_results,
)


display_regime_table(
    "SESSION BLOCK",
    session_results,
)


display_regime_table(
    "BUY / SELL",
    side_results,
)


display_regime_table(
    "TREND × VOLATILITY",
    trend_vol_results,
)


# ============================================================
# 24. 年安定性
# ============================================================

print()

print(
    "===================================="
)

print(
    "TREND YEAR STABILITY"
)

print(
    "===================================="
)

print(
    trend_stability.to_string(
        index=False
    )
)


print()

print(
    "===================================="
)

print(
    "VOL YEAR STABILITY"
)

print(
    "===================================="
)

print(
    vol_stability.to_string(
        index=False
    )
)


print()

print(
    "===================================="
)

print(
    "SESSION YEAR STABILITY"
)

print(
    "===================================="
)

print(
    session_stability.to_string(
        index=False
    )
)


print()

print(
    "===================================="
)

print(
    "SIDE YEAR STABILITY"
)

print(
    "===================================="
)

print(
    side_stability.to_string(
        index=False
    )
)


# ============================================================
# 25. Bootstrap
# ============================================================

bootstrap_show = (
    bootstrap_results.copy()
)


for col in [

    "bootstrap_mean",
    "ci_2_5",
    "ci_97_5",
    "prob_mean_positive",

]:

    bootstrap_show[
        col
    ] *= 100


print()

print(
    "===================================="
)

print(
    "MOVING BLOCK BOOTSTRAP"
)

print(
    "===================================="
)

print(
    bootstrap_show.to_string(
        index=False
    )
)


# ============================================================
# 26. Leave-One-Out
# ============================================================

leave_show = (
    leave_one_out_results.copy()
)


for col in [

    "net_win_rate",
    "avg_return",
    "median_return",
    "max_dd",
    "growth",
    "sum_return",

]:

    if col in (
        leave_show.columns
    ):

        leave_show[
            col
        ] *= 100


print()

print(
    "===================================="
)

print(
    "LEAVE ONE REGIME OUT"
)

print(
    "===================================="
)

print(
    leave_show.to_string(
        index=False
    )
)


# ============================================================
# 27. 最重要な診断メッセージ
# ============================================================

print()

print(
    "===================================="
)

print(
    "AUTOMATIC DIAGNOSTIC"
)

print(
    "===================================="
)


evaluated_years = (
    len(
        annual_results
    )
)


positive_years = (
    annual_results[
        "avg_return"
    ]
    >
    0
).sum()


pf_positive_years = (
    annual_results[
        "profit_factor"
    ]
    >
    1
).sum()


print(
    "Corrected evaluated years:",
    evaluated_years
)

print(
    "Positive years:",
    positive_years,
    "/",
    evaluated_years
)

print(
    "PF > 1 years:",
    pf_positive_years,
    "/",
    evaluated_years
)

print(
    "Corrected overall PF:",
    overall[
        "profit_factor"
    ]
)


if (
    overall[
        "profit_factor"
    ]
    <=
    1
):

    print()

    print(
        "WARNING:"
    )

    print(
        "境界・欠測修正後に統合PF <= 1です。"
    )

    print(
        "この場合、TP/SL最適化へ進まず、"
    )

    print(
        "まず元のedge仮説を再検討してください。"
    )

else:

    print()

    print(
        "Corrected baseline still has PF > 1."
    )

    print(
        "次にRegime表を見て、"
    )

    print(
        "利益が一部の市場状態だけに集中していないか確認します。"
    )


# ============================================================
# 28. グラフ
# ============================================================

# ----------------------------------------
# Corrected annual PF
# ----------------------------------------

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    annual_results[
        "test_year"
    ],
    annual_results[
        "profit_factor"
    ],
    marker="o",
)

plt.axhline(
    1,
    linewidth=1,
)

plt.xlabel(
    "Test Year"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Corrected Nested Walk-Forward PF"
)

plt.tight_layout()

plt.show()


# ----------------------------------------
# Trend PF
# ----------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

plt.bar(
    trend_results[
        "trend_regime"
    ],
    trend_results[
        "profit_factor"
    ],
)

plt.axhline(
    1,
    linewidth=1,
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "OOS PF by Trend Regime"
)

plt.tight_layout()

plt.show()


# ----------------------------------------
# Volatility PF
# ----------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

plt.bar(
    vol_results[
        "vol_regime"
    ],
    vol_results[
        "profit_factor"
    ],
)

plt.axhline(
    1,
    linewidth=1,
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "OOS PF by Volatility Regime"
)

plt.tight_layout()

plt.show()


# ============================================================
# 29. 保存
# ============================================================

annual_results.to_csv(
    OUTPUT_DIR
    /
    "corrected_annual_results.csv",
    index=False,
)


trades.to_csv(
    OUTPUT_DIR
    /
    "corrected_oos_trades.csv",
)


regime_trades.to_csv(
    OUTPUT_DIR
    /
    "oos_trades_with_regimes.csv",
)


regime_cutoffs.to_csv(
    OUTPUT_DIR
    /
    "regime_cutoffs_by_year.csv",
    index=False,
)


trend_results.to_csv(
    OUTPUT_DIR
    /
    "trend_results.csv",
    index=False,
)


vol_results.to_csv(
    OUTPUT_DIR
    /
    "volatility_results.csv",
    index=False,
)


session_results.to_csv(
    OUTPUT_DIR
    /
    "session_results.csv",
    index=False,
)


side_results.to_csv(
    OUTPUT_DIR
    /
    "side_results.csv",
    index=False,
)


trend_vol_results.to_csv(
    OUTPUT_DIR
    /
    "trend_vol_cross.csv",
    index=False,
)


trend_yearly.to_csv(
    OUTPUT_DIR
    /
    "trend_yearly.csv",
    index=False,
)


vol_yearly.to_csv(
    OUTPUT_DIR
    /
    "vol_yearly.csv",
    index=False,
)


session_yearly.to_csv(
    OUTPUT_DIR
    /
    "session_yearly.csv",
    index=False,
)


side_yearly.to_csv(
    OUTPUT_DIR
    /
    "side_yearly.csv",
    index=False,
)


trend_stability.to_csv(
    OUTPUT_DIR
    /
    "trend_stability.csv",
    index=False,
)


vol_stability.to_csv(
    OUTPUT_DIR
    /
    "vol_stability.csv",
    index=False,
)


session_stability.to_csv(
    OUTPUT_DIR
    /
    "session_stability.csv",
    index=False,
)


side_stability.to_csv(
    OUTPUT_DIR
    /
    "side_stability.csv",
    index=False,
)


bootstrap_results.to_csv(
    OUTPUT_DIR
    /
    "block_bootstrap.csv",
    index=False,
)


leave_one_out_results.to_csv(
    OUTPUT_DIR
    /
    "leave_one_regime_out.csv",
    index=False,
)


if validation_tables:

    pd.concat(
        validation_tables,
        ignore_index=True
    ).to_csv(

        OUTPUT_DIR
        /
        "validation_threshold_search.csv",

        index=False,
    )


print()

print(
    "===================================="
)

print(
    "FINISHED"
)

print(
    "===================================="
)

print(
    "Results saved to:"
)

print(
    OUTPUT_DIR.resolve()
)

print()

print(
    "最重要な確認順:"
)

print(
    "1. CORRECTED NESTED BASELINE"
)

print(
    "2. TREND REGIME"
)

print(
    "3. VOLATILITY REGIME"
)

print(
    "4. TREND × VOLATILITY"
)

print(
    "5. YEAR STABILITY"
)

print(
    "6. MOVING BLOCK BOOTSTRAP"
)

print(
    "7. LEAVE ONE REGIME OUT"
)
